# 🚀 RON AI — Qwen 2.5 0.5B Fine-Tuning & GGUF Exporter
This notebook fine-tunes **Qwen 2.5 0.5B-Instruct** using **Unsloth (QLoRA)** and exports it to a high-speed **GGUF (Q4_K_M)** file ready for local **Ollama** and 100% offline desktop voice control.

### ⚡ Hardware Requirement:
- Ensure your runtime is **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
# Step 1: Install Unsloth & dependencies
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes datasets

In [ ]:
# Step 2: Upload ron_train_dataset.jsonl
from google.colab import files
import os

if not os.path.exists("ron_train_dataset.jsonl"):
    print("Please select and upload 'ron_train_dataset.jsonl' from your local computer:")
    uploaded = files.upload()
else:
    print("Found ron_train_dataset.jsonl ready on disk!")

In [ ]:
# Step 3: Load Qwen 2.5 0.5B in 4-bit and apply LoRA
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model & LoRA configured!")

In [ ]:
# Step 4: Format dataset with ChatML template
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files="ron_train_dataset.jsonl", split="train")

def format_chatml(batch):
    formatted = []
    for conv in batch["messages"]:
        text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

dataset = raw_dataset.map(format_chatml, batched=True)
print(f"Loaded {len(dataset)} training examples.")

In [ ]:
# Step 5: Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        output_dir="ron_checkpoint",
    ),
)

trainer.train()
print("Training complete!")

In [ ]:
# Step 6: Export to GGUF format (Q4_K_M)
model.save_pretrained_gguf("ron_qwen_0.5b", tokenizer, quantization_method="q4_k_m")
print("Exported GGUF!")

In [ ]:
# Step 7: Download the GGUF file to your PC
import glob
files_list = glob.glob("*Q4_K_M*.gguf") + glob.glob("ron_qwen_0.5b/*Q4_K_M*.gguf")
if files_list:
    target = files_list[0]
    print(f"Downloading {target} to your PC...")
    files.download(target)
else:
    print("Look for the .gguf file in the file explorer on the left and right-click -> Download.")